In [ ]:
# !python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [1]:
import os, json, time, requests
import pandas as pd
from tqdm import tqdm
from woc.remote import WocMapsRemote

netid = 'ttb615'
woc = WocMapsRemote(base_url="https://worldofcode.org/api/")

projects = ['OHDSI/WebAPI', 'fkong7/2dunet', 'maxulysse/nf-core_modules',
            'opensafely/long-covid-risk-factors-and-prediction', 'omniscale/imposm3',
            'fcambus/telize', 'ziotom78/Healpix.jl', 'mpmccode/mpmc',
            'hyperopt/hyperopt-sklearn', 'geodynamics/aspect']

rows = []
for repo in projects:
    prj = repo.lower().replace('/', '_', 2)   # WoC project id
    shas = woc.get_values('p2c', prj)
    print(prj, len(shas))
    rows += [(prj, s) for s in shas]
df = pd.DataFrame(rows, columns=['project', 'sha1'])
df.to_csv('df_commits.csv', index=False)
df.head(2)

ohdsi_webapi 7903
fkong7_2dunet 397
maxulysse_nf-core_modules 8188
opensafely_long-covid-risk-factors-and-prediction 587
omniscale_imposm3 1276
fcambus_telize 342
ziotom78_healpix.jl 745
mpmccode_mpmc 654
hyperopt_hyperopt-sklearn 661
geodynamics_aspect 23608


,project,sha1
0,ohdsi_webapi,00039d9b4d4f15e202b8c4f9add59b1e73cc2128
1,ohdsi_webapi,000cc5c67e9661bb26b6e487540f11c927453338


In [2]:
CK = 'commit_data.jsonl'
done = set()
if os.path.exists(CK):
    with open(CK) as f:
        done = {json.loads(l)['commit'] for l in f}

todo = [s for s in df['sha1'].unique() if s not in done]
print(len(todo), 'commits to fetch')
errors = []

with open(CK, 'a') as f:
    for i in tqdm(range(0, len(todo), 10)):
        chunk = todo[i:i+10]
        for attempt in range(3): 
            try:
                res, err = woc.get_values_many('commit.tch', chunk)
                break
            except Exception as e:
                print('retry', attempt, e); time.sleep(5)
        else:
            errors.append(chunk); continue
        if err:
            errors.append(err); print('Got Errors', err)
        for sha, v in res.items():
            c = v[0]
            f.write(json.dumps({'commit': sha, 'author': c[2][0],
                                'time': int(c[2][1]), 'message': c[4]}) + '\n')
        f.flush()
        time.sleep(1)
print('chunks with errors:', len(errors))

44361 commits to fetch


  0%|▋                                                                                                                                                                                                 | 17/4437 [00:17<1:17:21,  1.05s/it]

Got Errors {'0575d15cf0468b545e8cdb9363d4b4e9e5ec9b9c': 'Key 0575d15cf0468b545e8cdb9363d4b4e9e5ec9b9c not found in /da5_fast/All.sha1c/commit_5.tch'}


  1%|█▊                                                                                                                                                                                                | 41/4437 [00:42<1:16:25,  1.04s/it]

Got Errors {'0dee6eaa98b281a4cd82df5300ca73a767b11625': 'Key 0dee6eaa98b281a4cd82df5300ca73a767b11625 not found in /da5_fast/All.sha1c/commit_13.tch'}


...

Got Errors {'ff6c60571730aa5b3a3fbba7bd0fa4e484db7316': 'Key ff6c60571730aa5b3a3fbba7bd0fa4e484db7316 not found in /da5_fast/All.sha1c/commit_127.tch'}


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋| 4431/4437 [1:17:06<00:06,  1.04s/it]

Got Errors {'ff7cab62c191ff7fa08d1b6e4b24755fcab642a3': 'Key ff7cab62c191ff7fa08d1b6e4b24755fcab642a3 not found in /da5_fast/All.sha1c/commit_127.tch'}


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4437/4437 [1:17:13<00:00,  1.04s/it]

chunks with errors: 166


In [3]:
cd = pd.read_json(CK, lines=True)
print(len(cd), 'commits saved of', df['sha1'].nunique(), 'expected')
missing = sorted(set(df['sha1']) - set(cd['commit']))
print('missing:', missing)
print(cd.iloc[0].to_dict())

44191 commits saved of 44361 expected
missing: ['031e880c57a5d223f6924d1a5f32731d6f59d9fc', '04d2b40f468d45b25ab7a24e1cf6bca84ac41f2f', '05049284bf087688c247e23c0f29a6c4fceb2a79', '056965efee7044b594b5c495a98ef864d33414ba', '0575d15cf0468b545e8cdb9363d4b4e9e5ec9b9c', '0991b910750ae6f5359d4c4ed3d266f6fad09adf', '0b5ee35e3db9d644a66e49ad19ea54859faaa7fa', '0ceca00033125ce24a66b72bfeec9556e92cda95', '0d08825ca92c0b75c7b79f5cff3ac6db1e0f07a9', '0d4482352ea6dcc2b8bc349d230fd5e6be4081a1', '0dbe4686dfbe80af63881e3a5b68e3a49a4187b4', '0dee6eaa98b281a4cd82df5300ca73a767b11625', '12a8d3623d6c0c9f76092a5051618821525b0c3e', '16ed05753b3402ea7ce6166c20e9ec550dfeddae', '17664b32cafe7f8b27e51aebcf277b19ed326fbe', '17d9f06b2fe67e0179ecdf497148cb13d82f10d6', '17efb9254607e099fc937ed7d507e28b22afdeec', '1a29041e5311a05f2308fcaf54c0fe804c8626e0', '1bf3ee44cf9a1a1a705d2aa3cd7e48ecef13439c', '1c6a7cc4e4ca4808640dc5edca669ca59aa59181', '1d86251e732977a935f4c010cd4c5fb62b6e664f', '24760c35ab3639635419ec5f98e

In [9]:
res2, err2 = woc.show_content_many('commit', missing)
print(err2)
for k, v in res2.items():
    print(k, '->', str(v)[:300])

{'031e880c57a5d223f6924d1a5f32731d6f59d9fc': 'Key 031e880c57a5d223f6924d1a5f32731d6f59d9fc not found in /da5_fast/All.sha1c/commit_3.tch', '04d2b40f468d45b25ab7a24e1cf6bca84ac41f2f': 'Key 04d2b40f468d45b25ab7a24e1cf6bca84ac41f2f not found in /da5_fast/All.sha1c/commit_4.tch', '05049284bf087688c247e23c0f29a6c4fceb2a79': 'Key 05049284bf087688c247e23c0f29a6c4fceb2a79 not found in /da5_fast/All.sha1c/commit_5.tch', '056965efee7044b594b5c495a98ef864d33414ba': 'Key 056965efee7044b594b5c495a98ef864d33414ba not found in /da5_fast/All.sha1c/commit_5.tch', '0575d15cf0468b545e8cdb9363d4b4e9e5ec9b9c': 'Key 0575d15cf0468b545e8cdb9363d4b4e9e5ec9b9c not found in /da5_fast/All.sha1c/commit_5.tch', '0991b910750ae6f5359d4c4ed3d266f6fad09adf': 'Key 0991b910750ae6f5359d4c4ed3d266f6fad09adf not found in /da5_fast/All.sha1c/commit_9.tch', '0b5ee35e3db9d644a66e49ad19ea54859faaa7fa': 'Key 0b5ee35e3db9d644a66e49ad19ea54859faaa7fa not found in /da5_fast/All.sha1c/commit_11.tch', '0ceca00033125ce24a66b72bfeec955

In [4]:
cd = pd.read_json(CK, lines=True)
out = df.merge(cd, left_on='sha1', right_on='commit')[['project', 'commit', 'author', 'time', 'message']]
out.to_csv(f'{netid}_project_summary.csv', index=False, sep=';', header=False)
print(len(df), 'SHAs listed;', len(out), 'rows written;', len(df) - len(out), 'missing')
out.head(2)


44361 SHAs listed; 44191 rows written; 170 missing


,project,commit,author,time,message
0,ohdsi_webapi,00039d9b4d4f15e202b8c4f9add59b1e73cc2128,SSStanleyZ <szea@spinsys.com>,1716211692,MDACA Spring Boot 3 migration\n
1,ohdsi_webapi,000cc5c67e9661bb26b6e487540f11c927453338,Chris Knoll <cknoll@ohdsi.org>,1544718025,Merge 214e50171f39e8d6bef573b866724de0c1263fbe...


In [5]:
chk = pd.read_csv(f'{netid}_project_summary.csv', sep=';', header=None,
                  names=['project_wocid', 'commit_sha1', 'author', 'time', 'commit message'])
print(len(chk), 'rows read back (expected', len(out), ')')
print(chk['project_wocid'].value_counts())
chk.head(2)

44191 rows read back (expected 44191 )
project_wocid
geodynamics_aspect                                   23494
maxulysse_nf-core_modules                             8188
ohdsi_webapi                                          7861
omniscale_imposm3                                     1270
ziotom78_healpix.jl                                    742
hyperopt_hyperopt-sklearn                              660
mpmccode_mpmc                                          654
opensafely_long-covid-risk-factors-and-prediction      587
fkong7_2dunet                                          397
fcambus_telize                                         338
Name: count, dtype: int64


,project_wocid,commit_sha1,author,time,commit message
0,ohdsi_webapi,00039d9b4d4f15e202b8c4f9add59b1e73cc2128,SSStanleyZ <szea@spinsys.com>,1716211692,MDACA Spring Boot 3 migration\n
1,ohdsi_webapi,000cc5c67e9661bb26b6e487540f11c927453338,Chris Knoll <cknoll@ohdsi.org>,1544718025,Merge 214e50171f39e8d6bef573b866724de0c1263fbe...


In [6]:
woc_stats = out.groupby('project').agg(ncommits=('commit', 'nunique'),
                                       nauthors=('author', 'nunique'),
                                       From=('time', 'min'), To=('time', 'max')).reset_index()
woc_stats['From_date'] = pd.to_datetime(woc_stats['From'], unit='s')
woc_stats['To_date'] = pd.to_datetime(woc_stats['To'], unit='s')
print(woc_stats.to_string(index=False))

                                          project  ncommits  nauthors       From         To           From_date             To_date
                                   fcambus_telize       338        13 1377084651 1750791632 2013-08-21 11:30:51 2025-06-24 19:00:32
                                    fkong7_2dunet       397         9 1553211731 1748543205 2019-03-21 23:42:11 2025-05-29 18:26:45
                               geodynamics_aspect     23494       347 1318199520 1776492642 2011-10-09 22:32:00 2026-04-18 06:10:42
                        hyperopt_hyperopt-sklearn       660        53 1361290194 1742634354 2013-02-19 16:09:54 2025-03-22 09:05:54
                        maxulysse_nf-core_modules      8188       619 1564131265 1761924543 2019-07-26 08:54:25 2025-10-31 15:29:03
                                    mpmccode_mpmc       654        43 1330027139 1761071129 2012-02-23 19:58:59 2025-10-21 18:25:29
                                     ohdsi_webapi      7861       211 141763

In [7]:
headers = {} 
gh = []
for repo in projects:
    info = requests.get(f'https://api.github.com/repos/{repo}', headers=headers).json()
    last = requests.get(f'https://api.github.com/repos/{repo}/commits?per_page=1', headers=headers).json()
    if isinstance(last, list) and last:
        date = last[0]['commit']['committer']['date'][:10].replace('-', '/')
    else:
        date = None
        print('problem with', repo, ':', info.get('message') or last)
    gh.append({'project': repo.lower().replace('/', '_', 2),
               'nstars': info.get('stargazers_count'),
               'nforks': info.get('forks_count'),
               'LastGHCommitDate': date})
gh = pd.DataFrame(gh)
print(gh.to_string(index=False))

                                          project  nstars  nforks LastGHCommitDate
                                     ohdsi_webapi     152     184       2026/09/28
                                    fkong7_2dunet      15       1       2025/02/28
                        maxulysse_nf-core_modules       1       0       2026/09/11
opensafely_long-covid-risk-factors-and-prediction       1       0       2024/11/26
                                omniscale_imposm3     774     161       2026/08/05
                                   fcambus_telize     873     138       2026/07/09
                              ziotom78_healpix.jl      58      20       2026/06/24
                                    mpmccode_mpmc      53      13       2025/10/21
                        hyperopt_hyperopt-sklearn    1647     273       2025/03/22
                               geodynamics_aspect     311     277       2026/09/28


In [8]:
stats = woc_stats[['project', 'ncommits', 'nauthors', 'From', 'To']].merge(gh, on='project')
stats.columns = ['project', '#commits', '#authors', 'From', 'To', '#stars', '#forks', 'LastGHCommitDate']
stats.to_csv(f'{netid}_project_stats.csv', sep=';', index=False)
stats

,project,#commits,#authors,From,To,#stars,#forks,LastGHCommitDate
0,fcambus_telize,338,13,1377084651,1750791632,873,138,2026/07/09
1,fkong7_2dunet,397,9,1553211731,1748543205,15,1,2025/02/28
2,geodynamics_aspect,23494,347,1318199520,1776492642,311,277,2026/09/28
3,hyperopt_hyperopt-sklearn,660,53,1361290194,1742634354,1647,273,2025/03/22
4,maxulysse_nf-core_modules,8188,619,1564131265,1761924543,1,0,2026/09/11
5,mpmccode_mpmc,654,43,1330027139,1761071129,53,13,2025/10/21
6,ohdsi_webapi,7861,211,1417636192,1762475208,152,184,2026/09/28
7,omniscale_imposm3,1270,71,1356527250,1762291878,774,161,2026/08/05
8,opensafely_long-covid-risk-factors-and-prediction,587,6,1636586992,1732624252,1,0,2024/11/26
9,ziotom78_healpix.jl,742,29,1419267990,1758788511,58,20,2026/06/24


In [10]:
s = stats.merge(woc_stats[['project', 'From_date', 'To_date']], on='project')
print("### GitHub stats\n| project | stars | forks | last commit date |\n|---|---|---|---|")
for _, r in s.iterrows():
    print(f"| {r['project']} | {r['#stars']} | {r['#forks']} | {r['LastGHCommitDate']} |")
print("\n### WoC stats\n| project | commits | authors | min time | max time |\n|---|---|---|---|---|")
for _, r in s.iterrows():
    print(f"| {r['project']} | {r['#commits']} | {r['#authors']} | {r['From_date']} | {r['To_date']} |")

### GitHub stats
| project | stars | forks | last commit date |
|---|---|---|---|
| fcambus_telize | 873 | 138 | 2026/07/09 |
| fkong7_2dunet | 15 | 1 | 2025/02/28 |
| geodynamics_aspect | 311 | 277 | 2026/09/28 |
| hyperopt_hyperopt-sklearn | 1647 | 273 | 2025/03/22 |
| maxulysse_nf-core_modules | 1 | 0 | 2026/09/11 |
| mpmccode_mpmc | 53 | 13 | 2025/10/21 |
| ohdsi_webapi | 152 | 184 | 2026/09/28 |
| omniscale_imposm3 | 774 | 161 | 2026/08/05 |
| opensafely_long-covid-risk-factors-and-prediction | 1 | 0 | 2024/11/26 |
| ziotom78_healpix.jl | 58 | 20 | 2026/06/24 |

### WoC stats
| project | commits | authors | min time | max time |
|---|---|---|---|---|
| fcambus_telize | 338 | 13 | 2013-08-21 11:30:51 | 2025-06-24 19:00:32 |
| fkong7_2dunet | 397 | 9 | 2019-03-21 23:42:11 | 2025-05-29 18:26:45 |
| geodynamics_aspect | 23494 | 347 | 2011-10-09 22:32:00 | 2026-04-18 06:10:42 |
| hyperopt_hyperopt-sklearn | 660 | 53 | 2013-02-19 16:09:54 | 2025-03-22 09:05:54 |
| maxulysse_nf-core_modu

### GitHub stats
| project | stars | forks | last commit date |
|---|---|---|---|
| fcambus_telize | 873 | 138 | 2026/07/09 |
| fkong7_2dunet | 15 | 1 | 2025/02/28 |
| geodynamics_aspect | 311 | 277 | 2026/09/28 |
| hyperopt_hyperopt-sklearn | 1647 | 273 | 2025/03/22 |
| maxulysse_nf-core_modules | 1 | 0 | 2026/09/11 |
| mpmccode_mpmc | 53 | 13 | 2025/10/21 |
| ohdsi_webapi | 152 | 184 | 2026/09/28 |
| omniscale_imposm3 | 774 | 161 | 2026/08/05 |
| opensafely_long-covid-risk-factors-and-prediction | 1 | 0 | 2024/11/26 |
| ziotom78_healpix.jl | 58 | 20 | 2026/06/24 |

### WoC stats
| project | commits | authors | min time | max time |
|---|---|---|---|---|
| fcambus_telize | 338 | 13 | 2013-08-21 11:30:51 | 2025-06-24 19:00:32 |
| fkong7_2dunet | 397 | 9 | 2019-03-21 23:42:11 | 2025-05-29 18:26:45 |
| geodynamics_aspect | 23494 | 347 | 2011-10-09 22:32:00 | 2026-04-18 06:10:42 |
| hyperopt_hyperopt-sklearn | 660 | 53 | 2013-02-19 16:09:54 | 2025-03-22 09:05:54 |
| maxulysse_nf-core_modules | 8188 | 619 | 2019-07-26 08:54:25 | 2025-10-31 15:29:03 |
| mpmccode_mpmc | 654 | 43 | 2012-02-23 19:58:59 | 2025-10-21 18:25:29 |
| ohdsi_webapi | 7861 | 211 | 2014-12-03 19:49:52 | 2025-11-07 00:26:48 |
| omniscale_imposm3 | 1270 | 71 | 2012-12-26 13:07:30 | 2025-11-04 21:31:18 |
| opensafely_long-covid-risk-factors-and-prediction | 587 | 6 | 2021-11-10 23:29:52 | 2024-11-26 12:30:52 |
| ziotom78_healpix.jl | 742 | 29 | 2014-12-22 17:06:30 | 2025-09-25 08:21:51 |

Notes: WoC listed 44,361 commit SHAs for these projects; 170 (0.4%) were not found in WoC's commit table and 
are excluded (ohdsi_webapi 42, geodynamics_aspect 114, omniscale_imposm3 6, fcambus_telize 4, 
ziotom78_healpix.jl 3, hyperopt_hyperopt-sklearn 1). Several WoC last-commit dates are later than the README's 
stated Jan 2025 cutoff. For fkong7_2dunet, WoC's latest commit (2025-05-29) is later than GitHub's last 
commit date (2025/02/28).

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github